# 07 - Final Resume Screener Demo

This notebook demonstrates the complete project pipeline:

1. Upload or paste a resume
2. Enter a job description
3. Extract matched and missing skills
4. Generate a BERT match score
5. Generate feedback using a free local Qwen model

No OpenAI API key is required.

Before running, select **Runtime > Change runtime type > T4 GPU**.

In [ ]:
# Install required libraries
!pip -q install transformers accelerate sentencepiece pdfplumber python-docx ipywidgets

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path

BASE_PATH = Path('/content/drive/MyDrive/AI-Resume-Screener')
BERT_MODEL_PATH = BASE_PATH / 'models/bert/final'

print('Project path:', BASE_PATH)
print('BERT model path:', BERT_MODEL_PATH)

In [ ]:
# Check GPU availability
import torch

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

print('CUDA available:', torch.cuda.is_available())
print('Device:', DEVICE)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'No GPU detected')

if not torch.cuda.is_available():
    print('Warning: BERT scoring and feedback generation will be slow without a GPU.')

## Load the Fine-Tuned BERT Match Model

Notebook 4 should have saved the trained model under:

`MyDrive/AI-Resume-Screener/models/bert/final/`

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

if not BERT_MODEL_PATH.exists():
    raise FileNotFoundError(
        f'BERT model not found at {BERT_MODEL_PATH}. Run Notebook 4 first.'
    )

bert_tokenizer = AutoTokenizer.from_pretrained(str(BERT_MODEL_PATH))
bert_model = AutoModelForSequenceClassification.from_pretrained(str(BERT_MODEL_PATH))
bert_model.to(DEVICE)
bert_model.eval()

print('Fine-tuned BERT model loaded successfully.')

## Load the Free Local Feedback Model

The smaller `Qwen2.5-0.5B-Instruct` model is used because BERT is loaded at the same time. You can replace it with `Qwen/Qwen2.5-1.5B-Instruct` if sufficient GPU memory is available.

In [ ]:
from transformers import AutoModelForCausalLM

FEEDBACK_MODEL_NAME = 'Qwen/Qwen2.5-0.5B-Instruct'

feedback_tokenizer = AutoTokenizer.from_pretrained(FEEDBACK_MODEL_NAME)
feedback_model = AutoModelForCausalLM.from_pretrained(
    FEEDBACK_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto',
)
feedback_model.eval()

print('Feedback model loaded successfully:', FEEDBACK_MODEL_NAME)

## Skill Extraction

In [ ]:
import json
import re

DEFAULT_SKILLS = [
    'python', 'java', 'javascript', 'typescript', 'c++', 'c#', 'go', 'rust',
    'sql', 'mysql', 'postgresql', 'mongodb', 'redis', 'oracle',
    'machine learning', 'deep learning', 'natural language processing', 'nlp',
    'computer vision', 'data analysis', 'data science', 'generative ai',
    'tensorflow', 'pytorch', 'scikit-learn', 'pandas', 'numpy', 'spark',
    'docker', 'kubernetes', 'git', 'linux', 'aws', 'azure', 'gcp', 'ci/cd',
    'react', 'angular', 'vue', 'node.js', 'django', 'flask', 'fastapi',
    'power bi', 'tableau', 'excel', 'project management', 'agile', 'scrum',
    'communication', 'leadership', 'problem solving', 'teamwork',
    'financial analysis', 'marketing', 'sales', 'recruitment',
]

custom_skills_path = BASE_PATH / 'models/skills.json'

if custom_skills_path.exists():
    with open(custom_skills_path, 'r', encoding='utf-8') as file:
        SKILLS = json.load(file)
    print('Loaded custom skill vocabulary:', len(SKILLS), 'skills')
else:
    SKILLS = DEFAULT_SKILLS
    print('Using default skill vocabulary:', len(SKILLS), 'skills')

def extract_skills(text):
    normalized = str(text).lower()
    return sorted({
        skill.lower()
        for skill in SKILLS
        if re.search(r'(?<!\\w)' + re.escape(skill) + r'(?!\\w)', normalized)
    })

def compare_skills(resume, job_description):
    resume_skills = set(extract_skills(resume))
    job_skills = set(extract_skills(job_description))
    matched = sorted(resume_skills & job_skills)
    missing = sorted(job_skills - resume_skills)
    coverage = len(matched) / len(job_skills) if job_skills else 0.0

    return {
        'resume_skills': sorted(resume_skills),
        'job_skills': sorted(job_skills),
        'matched_skills': matched,
        'missing_skills': missing,
        'skill_coverage': coverage,
    }

## BERT Match Scoring

In [ ]:
def get_bert_match_score(resume, job_description):
    encoded = bert_tokenizer(
        str(resume),
        str(job_description),
        truncation=True,
        max_length=512,
        return_tensors='pt',
    )
    encoded = {key: value.to(DEVICE) for key, value in encoded.items()}

    with torch.no_grad():
        score = bert_model(**encoded).logits.reshape(-1)[0].item()

    return max(0.0, min(1.0, score))

def calculate_overall_score(bert_score, skill_coverage):
    # BERT is the primary signal; explicit skill coverage adds explainability.
    return max(0.0, min(1.0, 0.80 * bert_score + 0.20 * skill_coverage))

## Local Feedback Generation

In [ ]:
FEW_SHOT_EXAMPLE = """
Example assessment:
Role: MLOps Engineer
Evidence: Python, Docker, Kubernetes, CI/CD, MLflow, and AWS projects.
Missing evidence: Terraform.
Assessment: Strong deployment and automation alignment, but infrastructure-as-code evidence should be added.
"""

def generate_feedback(resume, job_description, analysis, max_new_tokens=400):
    messages = [
        {
            'role': 'system',
            'content': (
                'You are an HR screening assistant. Use only evidence explicitly '
                'present in the resume and job description. Do not invent experience, '
                'qualifications, or personal characteristics. The output supports a '
                'human reviewer and must not make a final hiring decision.'
            ),
        },
        {
            'role': 'user',
            'content': f"""
{FEW_SHOT_EXAMPLE}

JOB DESCRIPTION:
{str(job_description)[:4000]}

RESUME:
{str(resume)[:4000]}

OVERALL MATCH SCORE:
{analysis['overall_score']:.2f}

BERT SCORE:
{analysis['bert_score']:.2f}

MATCHED SKILLS:
{analysis['matched_skills']}

MISSING SKILLS:
{analysis['missing_skills']}

Write concise feedback using exactly these headings:
Overall Match
Strengths
Matched Skills
Missing Skills
Recommendations
Limitations
""",
        },
    ]

    prompt = feedback_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = feedback_tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,
        max_length=6000,
    ).to(feedback_model.device)

    with torch.no_grad():
        output = feedback_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.08,
            pad_token_id=feedback_tokenizer.eos_token_id,
        )

    generated_tokens = output[0][inputs['input_ids'].shape[1]:]
    return feedback_tokenizer.decode(generated_tokens, skip_special_tokens=True).strip()

## Complete Analysis Pipeline

In [ ]:
def analyze_resume(resume, job_description, create_feedback=True):
    if not str(resume).strip() or not str(job_description).strip():
        raise ValueError('Both resume and job description are required.')

    skill_result = compare_skills(resume, job_description)
    bert_score = get_bert_match_score(resume, job_description)
    overall_score = calculate_overall_score(
        bert_score,
        skill_result['skill_coverage'],
    )

    result = {
        **skill_result,
        'bert_score': bert_score,
        'overall_score': overall_score,
        'resume_word_count': len(str(resume).split()),
        'job_word_count': len(str(job_description).split()),
    }

    if create_feedback:
        result['feedback'] = generate_feedback(resume, job_description, result)

    return result

## Upload a Resume Document

Supported formats: PDF, DOCX, and TXT. Skip this section if you prefer to paste resume text.

In [ ]:
from google.colab import files
from docx import Document
import pdfplumber

def upload_resume_document():
    uploaded = files.upload()
    if not uploaded:
        return ''

    filename = next(iter(uploaded))
    suffix = Path(filename).suffix.lower()

    if suffix == '.pdf':
        pages = []
        with pdfplumber.open(filename) as pdf:
            for page in pdf.pages:
                text = page.extract_text()
                if text:
                    pages.append(text)
        return '\n'.join(pages).strip()

    if suffix == '.docx':
        document = Document(filename)
        return '\n'.join(p.text for p in document.paragraphs).strip()

    if suffix == '.txt':
        with open(filename, 'r', encoding='utf-8', errors='replace') as file:
            return file.read().strip()

    raise ValueError('Unsupported file type. Upload PDF, DOCX, or TXT.')

# Uncomment the following line to upload a resume:
# resume_text = upload_resume_document()

## Enter Resume and Job Description

Replace the sample values below or use the upload function.

In [ ]:
resume_text = """
Machine Learning Engineer with experience developing NLP applications using
Python and PyTorch. Built REST APIs with FastAPI and deployed models using
Docker. Worked with SQL databases, Git, Linux, and AWS.
"""

job_description = """
We are hiring an NLP Engineer with strong Python, PyTorch, transformers,
Docker, Kubernetes, SQL, AWS, FastAPI, and Linux experience. The candidate
should communicate effectively and work with cross-functional teams.
"""

print('Resume words:', len(resume_text.split()))
print('Job description words:', len(job_description.split()))

## Run Final Analysis

In [ ]:
result = analyze_resume(
    resume=resume_text,
    job_description=job_description,
    create_feedback=True,
)

print(f"Overall Match Score: {result['overall_score'] * 100:.1f}%")
print(f"BERT Score: {result['bert_score'] * 100:.1f}%")
print(f"Skill Coverage: {result['skill_coverage'] * 100:.1f}%")
print('\nMatched Skills:', ', '.join(result['matched_skills']) or 'None detected')
print('Missing Skills:', ', '.join(result['missing_skills']) or 'None detected')
print('\n' + '=' * 80)
print('GENERATED FEEDBACK')
print('=' * 80 + '\n')
print(result['feedback'])

## Display Results Visually

In [ ]:
import matplotlib.pyplot as plt

labels = ['Overall Match', 'BERT', 'Skill Coverage']
values = [
    result['overall_score'] * 100,
    result['bert_score'] * 100,
    result['skill_coverage'] * 100,
]

plt.figure(figsize=(8, 4))
bars = plt.barh(labels, values, color=['#176b4d', '#40709b', '#d59b49'])
plt.xlim(0, 100)
plt.xlabel('Score (%)')
plt.title('Resume and Job Match Analysis')

for bar, value in zip(bars, values):
    plt.text(value + 1, bar.get_y() + bar.get_height() / 2, f'{value:.1f}%', va='center')

plt.show()

## Save the Analysis Report

In [ ]:
from datetime import datetime

OUTPUT_PATH = BASE_PATH / 'outputs'
OUTPUT_PATH.mkdir(parents=True, exist_ok=True)

export_result = {
    'created_at': datetime.now().isoformat(),
    'overall_score': result['overall_score'],
    'bert_score': result['bert_score'],
    'skill_coverage': result['skill_coverage'],
    'matched_skills': result['matched_skills'],
    'missing_skills': result['missing_skills'],
    'resume_skills': result['resume_skills'],
    'job_skills': result['job_skills'],
    'resume_word_count': result['resume_word_count'],
    'job_word_count': result['job_word_count'],
    'feedback': result['feedback'],
}

report_path = OUTPUT_PATH / 'latest_demo_analysis.json'
with open(report_path, 'w', encoding='utf-8') as file:
    json.dump(export_result, file, indent=2)

print('Saved report to:', report_path)

## Optional: Interactive Text Input

Run this cell to paste different resume and job-description text without editing previous cells.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

resume_widget = widgets.Textarea(
    value='',
    placeholder='Paste resume text here...',
    description='Resume:',
    layout=widgets.Layout(width='100%', height='220px'),
)

jd_widget = widgets.Textarea(
    value='',
    placeholder='Paste job description here...',
    description='Job:',
    layout=widgets.Layout(width='100%', height='220px'),
)

analyze_button = widgets.Button(
    description='Analyze Match',
    button_style='success',
)

interactive_output = widgets.Output()

def analyze_widget_input(_):
    with interactive_output:
        clear_output()
        try:
            analysis = analyze_resume(resume_widget.value, jd_widget.value)
            print(f"Overall Match: {analysis['overall_score'] * 100:.1f}%")
            print(f"BERT Score: {analysis['bert_score'] * 100:.1f}%")
            print(f"Skill Coverage: {analysis['skill_coverage'] * 100:.1f}%")
            print('\nMatched:', ', '.join(analysis['matched_skills']) or 'None')
            print('Missing:', ', '.join(analysis['missing_skills']) or 'None')
            print('\nFeedback:\n')
            print(analysis['feedback'])
        except Exception as error:
            print('Analysis failed:', error)

analyze_button.on_click(analyze_widget_input)

display(resume_widget, jd_widget, analyze_button, interactive_output)

## Demo Limitations

- BERT accepts a maximum of 512 tokens, so long resumes and job descriptions are truncated.
- Skill extraction depends on the configured skill vocabulary.
- Generated feedback may contain mistakes and requires human review.
- Match scores support comparison and must not be treated as final hiring decisions.
- The model should not be used to infer protected characteristics or automatically reject candidates.